# Load CSV

In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
def predict_file(csv_path, model_path=MODEL_OUT):
    """Dự đoán presence cho 1 file CSV mới."""
    model = CatBoostClassifier()
    model.load_model(model_path)

    buf_amp, buf_phase, buf_meta = [], [], None
    results = []

    for chunk in pd.read_csv(csv_path, chunksize=CHUNKSIZE):
        for col, val in DEFAULTS.items():
            if col not in chunk.columns:
                chunk[col] = val
        chunk.fillna(DEFAULTS, inplace=True)

        for _, row in chunk.iterrows():
            try:
                amp, phase = parse_iq(row['data'])
            except Exception:
                continue

            buf_amp.append(amp)
            buf_phase.append(phase)
            if buf_meta is None:
                buf_meta = row

            if len(buf_amp) == WINDOW:
                amp_arr   = np.array(buf_amp,   dtype=np.float32)
                phase_arr = np.array(buf_phase, dtype=np.float32)
                feat = extract_features(amp_arr, phase_arr, buf_meta)
                feat.pop('label', None)
                feat.pop('file_id', None)
                results.append(feat)
                buf_amp   = buf_amp[OVERLAP:]
                buf_phase = buf_phase[OVERLAP:]
                buf_meta  = None

    if not results:
        print("Không đủ dữ liệu.")
        return

    feat_df = pd.DataFrame(results)
    preds = model.predict(feat_df)
    probs = model.predict_proba(feat_df)[:, 1]

    presence_ratio = preds.mean()
    print(f"Có người: {presence_ratio*100:.1f}% số windows")
    print(f"Prob trung bình: {probs.mean():.3f}")
    return preds, probs


NameError: name 'MODEL_OUT' is not defined

# Calculate Amplitude and phase

In [ ]:
import ast
import numpy as np

# Parse raw I/Q stored in `data` as [I0, Q0, I1, Q1, ...]
def _parse_iq(value):
    if isinstance(value, str):
        value = ast.literal_eval(value)
    arr = np.asarray(value, dtype=float)
    if arr.size % 2 != 0:
        raise ValueError("data length must be even (I/Q pairs)")
    i = arr[0::2]
    q = arr[1::2]
    return i, q


def _drop_subcarriers(i, q, n_sub):
    if n_sub == 64:
        drop = list(range(0, 6)) + list(range(59, 64)) + [32]
    elif n_sub == 128:
        drop = list(range(0, 6)) + list(range(122, 128)) + [63, 64, 65]
    elif n_sub == 256:
        drop = list(range(0, 11)) + list(range(245, 256)) + [127, 128, 129]
    else:
        return i, q

    mask = np.ones(n_sub, dtype=bool)
    mask[drop] = False
    return i[mask], q[mask]


def _filter_row(row):
    i, q = _parse_iq(row["data"])
    n_sub = int(row["len"]) // 2
    i, q = _drop_subcarriers(i, q, n_sub)
    out = np.empty(i.size + q.size, dtype=float)
    out[0::2] = i
    out[1::2] = q
    return out

# Filter I/Q per row based on the subcarrier count in `len`
df["data_filtered"] = df.apply(_filter_row, axis=1)

df[["len", "data_filtered"]].head()

In [ ]:
import ast
import numpy as np
from concurrent.futures import ThreadPoolExecutor

# Parse raw I/Q stored in `data` as [I0, Q0, I1, Q1, ...]
def _parse_iq(value):
    if isinstance(value, str):
        value = ast.literal_eval(value)
    arr = np.asarray(value, dtype=float)
    if arr.size % 2 != 0:
        raise ValueError("data length must be even (I/Q pairs)")
    i = arr[0::2]
    q = arr[1::2]
    return i, q

# Compute amplitude/phase for a chunk of rows
# Returns lists of np arrays (one per row)
def _compute_chunk(series):
    amps = []
    phases = []
    for value in series:
        i, q = _parse_iq(value)
        
        # 1. Tinh Bien do va Pha tho
        amp = np.hypot(i, q)
        phase_raw = np.arctan2(q, i)
        
        amps.append(amp)
        phases.append(phase_raw)
        
    return amps, phases


# Split work by chunks for multithreading
n_chunks = max(1, (len(df) // 2000) + 1)
chunks = np.array_split(df["data_filtered"], n_chunks)

with ThreadPoolExecutor() as ex:
    results = list(ex.map(_compute_chunk, chunks))

amplitude = [amp for r in results for amp in r[0]]
phase = [ph for r in results for ph in r[1]]

# Store arrays per row
df["amplitude"] = amplitude
df["phase"] = phase

df[["amplitude", "phase"]].head()

# Apply hampel filter

In [ ]:
from hampel_filter import hampel_filter as hf
import numpy as np

def _hampel_1d(arr, window_size=20, n_sigma=3.0):
    arr = np.asarray(arr, dtype=float)
    if arr.size == 0:
        return arr
    outlier_indices = hf.hampel(
        arr,
        window_size=window_size,
        n=n_sigma,
        parallel=True,
        return_indices=True,
    )
    if isinstance(outlier_indices, tuple):
        outlier_indices = outlier_indices[0]
    outlier_indices = np.asarray(outlier_indices, dtype=int)
    
    if outlier_indices.size == 0:
        return arr
    
    filtered = arr.copy()
    for idx in outlier_indices:
        start = max(0, idx - window_size)
        end = min(arr.size, idx + window_size + 1)
        
        window = np.concatenate([arr[start:idx], arr[idx+1:end]])
        filtered[idx] = np.median(window)
    return filtered

df["amplitude_hampel"] = df["amplitude"].apply(_hampel_1d)
df["phase_hampel"] = df["phase"].apply(_hampel_1d)

df[["amplitude_hampel", "phase_hampel"]].head()

# Phase Unwarp and sanitization

In [ ]:
import numpy as np

def _unwrap_and_sanitize(arr):
    arr = np.asarray(arr, dtype=float)
    if arr.size < 2:
        return arr
    phase_unwrapped = np.unwrap(arr)
    k = np.arange(phase_unwrapped.size)
    a, b = np.polyfit(k, phase_unwrapped, 1)
    return phase_unwrapped - (a * k + b)

df["phase_raw"] = df["phase"]
df["phase"] = df["phase_raw"].apply(_unwrap_and_sanitize)

df[["phase_raw", "phase"]].head()

# Apply SG filter

In [ ]:
import numpy as np
from scipy.signal import savgol_filter

def _savgol_1d(arr, window_length=31, polyorder=3):
    arr = np.asarray(arr, dtype=float)
    if arr.size < 3:
        return arr
    wl = min(window_length, arr.size)
    if wl % 2 == 0:
        wl -= 1
    if wl < 3:
        return arr
    po = min(polyorder, wl - 1)
    return savgol_filter(arr, window_length=wl, polyorder=po)

amp_src = "amplitude_hampel" if "amplitude_hampel" in df.columns else "amplitude"
phase_src = "phase_hampel" if "phase_hampel" in df.columns else "phase"

df["amplitude_sg"] = df[amp_src].apply(_savgol_1d)
df["phase_sg"] = df[phase_src].apply(_savgol_1d)

df[[amp_src, "amplitude_sg", phase_src, "phase_sg"]].head()

# Apply Butterworth filter

In [ ]:
import numpy as np
from scipy.signal import butter, filtfilt

def _butter_1d(arr, fs=100.0, btype='band'):
    lowcut = 0.1
    highcut = 0.5
    order = 4

    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq

    arr = np.asarray(arr, dtype=float)
    if arr.size < (order * 3 + 1):
        return arr
    
    b, a = butter(order, [low, high], btype=btype, fs=fs)
    return filtfilt(b, a, arr)

amp_src = "amplitude_sg" if "amplitude_sg" in df.columns else "amplitude"
phase_src = "phase_sg" if "phase_sg" in df.columns else "phase"

df["amplitude_bw"] = df[amp_src].apply(_butter_1d)
df["phase_bw"] = df[phase_src].apply(_butter_1d)

df[[amp_src, "amplitude_bw", phase_src, "phase_bw"]].head()

# Visualize data before and after preprocessed

In [ ]:
def predict_file(csv_path, model_path=MODEL_OUT):
    """Dự đoán presence cho 1 file CSV mới."""
    model = CatBoostClassifier()
    model.load_model(model_path)

    buf_amp, buf_phase, buf_meta = [], [], None
    results = []

    for chunk in pd.read_csv(csv_path, chunksize=CHUNKSIZE):
        for col, val in DEFAULTS.items():
            if col not in chunk.columns:
                chunk[col] = val
        chunk.fillna(DEFAULTS, inplace=True)

        for _, row in chunk.iterrows():
            try:
                amp, phase = parse_iq(row['data'])
            except Exception:
                continue

            buf_amp.append(amp)
            buf_phase.append(phase)
            if buf_meta is None:
                buf_meta = row

            if len(buf_amp) == WINDOW:
                amp_arr   = np.array(buf_amp,   dtype=np.float32)
                phase_arr = np.array(buf_phase, dtype=np.float32)
                feat = extract_features(amp_arr, phase_arr, buf_meta)
                feat.pop('label', None)
                feat.pop('file_id', None)
                results.append(feat)
                buf_amp   = buf_amp[OVERLAP:]
                buf_phase = buf_phase[OVERLAP:]
                buf_meta  = None

    if not results:
        print("Không đủ dữ liệu.")
        return

    feat_df = pd.DataFrame(results)
    preds = model.predict(feat_df)
    probs = model.predict_proba(feat_df)[:, 1]

    presence_ratio = preds.mean()
    print(f"Có người: {presence_ratio*100:.1f}% số windows")
    print(f"Prob trung bình: {probs.mean():.3f}")
    return preds, probs